# Consistency Analysis: Lap-to-Lap Variation

This notebook analyzes your driving consistency by measuring variation in braking points, corner speeds, and throttle application across all laps.

## What You'll Find Here

- **Braking Point Consistency**: Box plots showing variation in where you start braking for each corner
- **Corner Speed Consistency**: Box plots showing minimum speed variation through each corner
- **Throttle Application Consistency**: Box plots showing where you get back on throttle at corner exits
- **Throttle Acceptance**: The lateral G at which you reach full throttle during corner exit, as a percentage of the corner's peak lateral G
- **Summary Statistics Table**: Mean, standard deviation, min, max, and range for each segment

## How to Interpret the Results

- **Tight box plots** (small range): Consistent performance - you're hitting the same marks each lap
- **Wide box plots** (large range): Inconsistent - opportunity for improvement through practice
- **Outliers** (dots outside whiskers): Unusual laps - could be mistakes, traffic, or experimenting with lines
- **High standard deviation**: Focus area for practice
- **High throttle acceptance %**: Getting on full throttle earlier while still cornering hard - more aggressive exit

## Using Your Own Data

To analyze your own data file:

1. **Upload your file**: Place your `.xrk` or `.xrz` file in the same directory as this notebook
2. **Update the filename**: In cell 4 below, change the filename:
   ```python
   log = aim_xrk("YOUR_FILENAME.xrz")  # Replace with your file
   ```
3. **Run all cells**: Execute the notebook from top to bottom

The notebook will automatically:
- Detect corners and zones from your GPS and pedal data
- Analyze all valid laps (excluding pit laps)
- Generate consistency metrics for each track segment

## Requirements

- GPS data channels (`GPS Latitude`, `GPS Longitude`, `GPS Speed`)
- Brake pressure (`BrakePress`) and throttle (`PPS`)
- Lateral acceleration (`LateralAcc`) for throttle acceptance analysis
- Multiple laps of data for meaningful consistency analysis

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [1]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import core libraries
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px

# Import libxrk:
from libxrk import aim_xrk

# Import helper functions
from motorsports_data_notebook import show_fig, get_best_lap, compute_lap_distance, identify_corners
from motorsports_data_notebook.helpers import (
    identify_zones_single_lap,
    average_zones_across_laps,
    merge_accel_zones_by_time,
    TrackSegment,
    create_track_segments,
)

In [3]:
log = aim_xrk("CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")

/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:334: RuntimeWarning: invalid value encountered in divide
  t = np.maximum(-np.sum(SN * O, axis=1) / np.sum(SN * D, axis=1), 0)
/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:347: RuntimeWarning: divide by zero encountered in divide
  t = np.maximum(-np.sum(SN * O, axis=1) / np.sum(SN * D, axis=1), 0)
/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:348: RuntimeWarning: invalid value encountered in multiply
  dist = np.sum(np.square(O + t.reshape((len(t), 1)) * D), axis=1)


In [4]:
# Flatten all the data into a uniform table, interpolating as needed (can use a lot of RAM!)
channels = log.get_channels_as_table().to_pandas()

# Add derived columns
channels["speed_kmh"] = channels["GPS Speed"] * 3.6

In [5]:
# Load laps and compute lap times
laps = log.laps.to_pandas()
laps["lap_time"] = pd.to_timedelta(laps["end_time"] - laps["start_time"], unit="ms")

# Compute distance_m for each lap and add to channels
# Distance resets at the start of each lap
channels["distance_m"] = 0.0

for idx, lap in laps.iterrows():
    lap_mask = (channels["timecodes"] >= lap["start_time"]) & (
        channels["timecodes"] <= lap["end_time"]
    )
    lap_indices = channels.index[lap_mask]

    if len(lap_indices) > 0:
        lap_timecodes = channels.loc[lap_indices, "timecodes"]
        lap_speed = channels.loc[lap_indices, "GPS Speed"]
        distance_values = compute_lap_distance(lap_timecodes.values, lap_speed.values)
        channels.loc[lap_indices, "distance_m"] = distance_values

laps.style.format(
    {"lap_time": lambda x: f"{int(x.total_seconds() // 60)}:{x.total_seconds() % 60:06.3f}"}
)

,num,start_time,end_time,lap_time
0,0,0,150454,2:30.454
1,1,150454,279602,2:09.148
2,2,279602,406240,2:06.638
3,3,406240,532797,2:06.557
4,4,532797,659282,2:06.485
5,5,659282,787773,2:08.491
6,6,787773,913776,2:06.003
7,7,913776,1041397,2:07.621
8,8,1041397,1168322,2:06.925
9,9,1168322,1294676,2:06.354


In [6]:
# Best lap extraction
best_lap = get_best_lap(laps)
start_ts = best_lap["start_time"]
end_ts = best_lap["end_time"]
# Use < for end_ts to exclude the first sample of the next lap (where distance resets to 0)
lap_channels = channels.query(f"timecodes >= @start_ts and timecodes < @end_ts").copy()

In [7]:
# Identify corners directly from GPS coordinates
corners = identify_corners(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    threshold=0.006,
    min_corner_length=15,
    min_gap=80,
)

print(f"Found {len(corners)} corners")

Found 9 corners


In [8]:
# Identify braking and acceleration zones averaged over top laps
# Get laps within 103% of best lap time
valid_laps = laps[laps["lap_time"] > pd.Timedelta(0)].copy()
best_lap_time = valid_laps["lap_time"].min()
threshold_time = best_lap_time * 1.03
top_laps = valid_laps[valid_laps["lap_time"] <= threshold_time]

# Collect zones from each top lap
all_braking_zones = []
all_accel_zones = []

for idx, lap in top_laps.iterrows():
    lap_start = lap["start_time"]
    lap_end = lap["end_time"]
    lap_data = channels.query(f"timecodes >= @lap_start and timecodes <= @lap_end").copy()

    if len(lap_data) < 10:
        continue

    braking, accel = identify_zones_single_lap(
        lap_data["distance_m"].values,
        lap_data["BrakePress"].values,
        lap_data["PPS"].values,
        lap_data["GPS Speed"].values,
    )
    all_braking_zones.append(braking)
    all_accel_zones.append(accel)

# Average zones across laps
braking_zones, accel_zones = average_zones_across_laps(
    all_braking_zones,
    all_accel_zones,
    track_length=lap_channels["distance_m"].max(),
    resolution=1.0,
    threshold=0.5,
)

# Merge acceleration zones separated by short time gaps (gear changes)
accel_zones = merge_accel_zones_by_time(
    accel_zones,
    braking_zones,
    lap_channels["distance_m"].values,
    lap_channels["GPS Speed"].values,
    max_gap_time=1.5,
)

print(f"Found {len(braking_zones)} braking zones and {len(accel_zones)} acceleration zones")

Found 7 braking zones and 7 acceleration zones


In [9]:
# Create track segments
track_length = lap_channels["distance_m"].iloc[-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

print(f"Created {len(segments)} track segments")

Created 27 track segments


In [10]:
# Extract all laps and compute per-lap segment statistics
def find_braking_point(lap_data, segment):
    """Find the distance where braking starts within a segment."""
    mask = (lap_data["distance_m"] >= segment.start_dist) & (
        lap_data["distance_m"] <= segment.end_dist
    )
    seg_data = lap_data[mask]

    if len(seg_data) == 0:
        return None

    # Find first point where brake > threshold
    brake_points = seg_data[seg_data["BrakePress"] > 5]
    if len(brake_points) > 0:
        return brake_points["distance_m"].iloc[0]
    return None


def find_throttle_point(lap_data, segment):
    """Find the distance where throttle application starts within a segment."""
    mask = (lap_data["distance_m"] >= segment.start_dist) & (
        lap_data["distance_m"] <= segment.end_dist
    )
    seg_data = lap_data[mask]

    if len(seg_data) == 0:
        return None

    # Find first point where throttle > threshold and brake < threshold
    throttle_points = seg_data[(seg_data["PPS"] > 20) & (seg_data["BrakePress"] < 5)]
    if len(throttle_points) > 0:
        return throttle_points["distance_m"].iloc[0]
    return None


def find_min_speed(lap_data, segment):
    """Find minimum speed within a segment (for corners)."""
    mask = (lap_data["distance_m"] >= segment.start_dist) & (
        lap_data["distance_m"] <= segment.end_dist
    )
    seg_data = lap_data[mask]

    if len(seg_data) == 0:
        return None

    return seg_data["speed_kmh"].min()


def compute_segment_stats_for_lap(lap_data, segments):
    """Compute statistics for each segment in a single lap."""
    stats = []

    for seg in segments:
        stat = {
            "segment_id": seg.id,
            "segment_name": seg.name,
            "segment_type": seg.segment_type,
            "corner_id": seg.corner_id,
        }

        if seg.segment_type == "braking":
            stat["braking_point"] = find_braking_point(lap_data, seg)
            # Calculate how early/late vs segment start
            if stat["braking_point"] is not None:
                stat["brake_offset"] = stat["braking_point"] - seg.start_dist

        elif seg.segment_type == "corner":
            stat["min_speed"] = find_min_speed(lap_data, seg)

        elif seg.segment_type == "acceleration":
            stat["throttle_point"] = find_throttle_point(lap_data, seg)
            if stat["throttle_point"] is not None:
                stat["throttle_offset"] = stat["throttle_point"] - seg.start_dist

        stats.append(stat)

    return stats


# Use only top laps (within 103% of best lap time) for consistency analysis
# This uses the same top_laps defined earlier for zone detection
print(f"Analyzing {len(top_laps)} laps (within 103% of best time)...")

# Compute statistics for each lap
all_lap_stats = []

for idx, lap in top_laps.iterrows():
    # Extract lap data (distance_m and speed_kmh already computed in channels)
    lap_data = channels.query(
        f'timecodes >= {lap["start_time"]} and timecodes <= {lap["end_time"]}'
    ).copy()

    if len(lap_data) < 10:
        continue

    # Compute segment stats
    lap_stats = compute_segment_stats_for_lap(lap_data, segments)

    for stat in lap_stats:
        stat["lap_num"] = lap["num"]
        stat["lap_time"] = lap["lap_time"]

    all_lap_stats.extend(lap_stats)

# Convert to DataFrame
stats_df = pd.DataFrame(all_lap_stats)
print(f"Computed {len(stats_df)} segment statistics across all laps")

Analyzing 13 laps (within 103% of best time)...
Computed 351 segment statistics across all laps
Computed 351 segment statistics across all laps


In [11]:
# Visualize braking consistency
# Show braking point variation for each corner, centered around the mean

braking_stats = stats_df[stats_df["segment_type"] == "braking"].dropna(subset=["braking_point"])

if len(braking_stats) > 0:
    # Calculate deviation from mean braking point for each segment
    braking_stats = braking_stats.copy()
    braking_stats["braking_deviation"] = braking_stats.groupby("segment_name")[
        "braking_point"
    ].transform(lambda x: x - x.mean())

    fig = px.box(
        braking_stats,
        x="segment_name",
        y="braking_deviation",
        title="Braking Point Consistency by Corner (Centered on Mean)",
        labels={"braking_deviation": "Deviation from Mean (m)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    # Add a reference line at zero (the mean)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    show_fig(fig)
else:
    print("No braking data available")

In [12]:
# Visualize corner minimum speed consistency
corner_stats = stats_df[stats_df["segment_type"] == "corner"].dropna(subset=["min_speed"])

if len(corner_stats) > 0:
    fig = px.box(
        corner_stats,
        x="segment_name",
        y="min_speed",
        title="Minimum Corner Speed Consistency",
        labels={"min_speed": "Min Speed (km/h)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No corner speed data available")

In [13]:
# Visualize throttle application consistency
# Show throttle point variation for each corner exit, centered around the mean

accel_stats = stats_df[stats_df["segment_type"] == "acceleration"].dropna(subset=["throttle_point"])

if len(accel_stats) > 0:
    # Calculate deviation from mean throttle point for each segment
    accel_stats = accel_stats.copy()
    accel_stats["throttle_deviation"] = accel_stats.groupby("segment_name")[
        "throttle_point"
    ].transform(lambda x: x - x.mean())

    fig = px.box(
        accel_stats,
        x="segment_name",
        y="throttle_deviation",
        title="Throttle Application Point Consistency (Centered on Mean)",
        labels={"throttle_deviation": "Deviation from Mean (m)", "segment_name": "Corner Exit"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    # Add a reference line at zero (the mean)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    show_fig(fig)
else:
    print("No throttle data available")

In [14]:
# Compute throttle acceptance for each corner across all laps
# Throttle acceptance = lateral G at sustained full throttle / peak lateral G of corner

from motorsports_data_notebook import find_throttle_acceptance

# Smoothing window for lateral G calculations and visualization
LATERAL_G_SMOOTHING_WINDOW = 25

throttle_acceptance_stats = []

for idx, lap in top_laps.iterrows():
    lap_data = channels.query(
        f'timecodes >= {lap["start_time"]} and timecodes <= {lap["end_time"]}'
    ).copy()

    if len(lap_data) < 10:
        continue

    for corner in corners:
        result = find_throttle_acceptance(lap_data, corner, smoothing_window=LATERAL_G_SMOOTHING_WINDOW)
        if result is not None:
            throttle_acceptance_stats.append({
                "corner_name": corner.name,
                "corner_id": corner.id,
                "lap_num": lap["num"],
                "throttle_acceptance_pct": result["throttle_acceptance_pct"],
                "lateral_g_at_throttle": result["lateral_g_at_throttle"],
                "peak_lateral_g": result["peak_lateral_g"],
                "full_throttle_dist": result["full_throttle_dist"],
            })

throttle_acceptance_df = pd.DataFrame(throttle_acceptance_stats)
print(f"Computed throttle acceptance for {len(throttle_acceptance_df)} corner/lap combinations")

Computed throttle acceptance for 94 corner/lap combinations


In [15]:
# Visualize throttle acceptance consistency
# Shows at what percentage of peak lateral G the driver reaches full throttle

if len(throttle_acceptance_df) > 0:
    fig = px.box(
        throttle_acceptance_df,
        x="corner_name",
        y="throttle_acceptance_pct",
        title="Throttle Acceptance by Corner (% of Peak Lateral G at Full Throttle)",
        labels={
            "throttle_acceptance_pct": "Throttle Acceptance (%)",
            "corner_name": "Corner",
        },
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    # Add reference lines
    fig.add_hline(y=100, line_dash="dash", line_color="red", opacity=0.5,
                  annotation_text="100% = Full throttle at peak G")
    show_fig(fig)
else:
    print("No throttle acceptance data available")

In [16]:
# Throttle acceptance summary statistics table
if len(throttle_acceptance_df) > 0:
    throttle_summary = []
    for corner_name in throttle_acceptance_df["corner_name"].unique():
        corner_data = throttle_acceptance_df[throttle_acceptance_df["corner_name"] == corner_name]
        throttle_summary.append({
            "Corner": corner_name,
            "Mean (%)": corner_data["throttle_acceptance_pct"].mean(),
            "Std (%)": corner_data["throttle_acceptance_pct"].std(),
            "Min (%)": corner_data["throttle_acceptance_pct"].min(),
            "Max (%)": corner_data["throttle_acceptance_pct"].max(),
            "Range (%)": corner_data["throttle_acceptance_pct"].max() - corner_data["throttle_acceptance_pct"].min(),
            "N": len(corner_data),
        })
    
    throttle_summary_df = pd.DataFrame(throttle_summary)
    display(throttle_summary_df.style.format({
        "Mean (%)": "{:.1f}",
        "Std (%)": "{:.1f}",
        "Min (%)": "{:.1f}",
        "Max (%)": "{:.1f}",
        "Range (%)": "{:.1f}",
    }))
else:
    print("No throttle acceptance data available")

,Corner,Mean (%),Std (%),Min (%),Max (%),Range (%),N
0,Turn 1,83.4,11.9,60.1,97.3,37.2,12
1,Turn 3,82.2,6.0,72.8,89.2,16.4,10
2,Turn 4,78.5,8.5,62.8,91.5,28.7,13
3,Turn 6,80.4,10.5,55.9,93.7,37.8,13
4,Turn 7,70.1,7.5,52.7,81.1,28.4,13
5,Turn 8,50.6,25.3,5.6,79.0,73.3,12
6,Turn 5,83.2,8.8,70.3,93.1,22.8,10
7,Turn 9,68.7,7.7,53.3,78.2,24.9,11


In [17]:
# Visualize throttle acceptance concept for Turn 1 (best lap)
# Shows throttle and lateral G over distance with reference lines

import plotly.graph_objects as go

# Get Turn 1 data
turn1 = corners[0]  # First corner
turn1_result = find_throttle_acceptance(lap_channels, turn1, smoothing_window=LATERAL_G_SMOOTHING_WINDOW)

if turn1_result is not None:
    # Extract data for the corner region (with some margin before/after)
    margin = 50  # meters before and after corner
    plot_mask = (lap_channels["distance_m"] >= turn1.start_dist - margin) & (
        lap_channels["distance_m"] <= turn1.end_dist + margin
    )
    plot_data = lap_channels[plot_mask].copy()
    
    # Apply smoothing to lateral G (rolling average) - use same window as calculations
    plot_data["LateralAcc_smooth"] = plot_data["LateralAcc"].abs().rolling(
        window=LATERAL_G_SMOOTHING_WINDOW, center=True, min_periods=1
    ).mean()
    
    fig = go.Figure()
    
    # Throttle trace (scaled to fit on same axis as G)
    fig.add_trace(go.Scatter(
        x=plot_data["distance_m"],
        y=plot_data["PPS"] / 100 * 2,  # Scale 0-100% to 0-2 for visibility
        mode="lines",
        name="Throttle (scaled)",
        line=dict(color="green", width=2),
        hovertemplate="Distance: %{x:.0f}m<br>Throttle: %{customdata:.0f}%<extra></extra>",
        customdata=plot_data["PPS"],
    ))
    
    # Lateral G trace (smoothed absolute value)
    fig.add_trace(go.Scatter(
        x=plot_data["distance_m"],
        y=plot_data["LateralAcc_smooth"],
        mode="lines",
        name="Lateral G (smoothed)",
        line=dict(color="blue", width=2),
        hovertemplate="Distance: %{x:.0f}m<br>Lateral G: %{y:.2f}G<extra></extra>",
    ))
    
    # Peak lateral G reference line
    fig.add_hline(
        y=turn1_result["peak_lateral_g"],
        line_dash="dash",
        line_color="blue",
        opacity=0.7,
        annotation_text=f"Peak Lateral G: {turn1_result['peak_lateral_g']:.2f}G",
        annotation_position="top right",
    )
    
    # Lateral G at throttle acceptance reference line
    fig.add_hline(
        y=turn1_result["lateral_g_at_throttle"],
        line_dash="dash",
        line_color="red",
        opacity=0.7,
        annotation_text=f"Lateral G at Full Throttle: {turn1_result['lateral_g_at_throttle']:.2f}G ({turn1_result['throttle_acceptance_pct']:.0f}%)",
        annotation_position="bottom right",
    )
    
    # Full throttle point vertical line
    fig.add_vline(
        x=turn1_result["full_throttle_dist"],
        line_dash="dash",
        line_color="green",
        opacity=0.7,
        annotation_text="Full Throttle Point",
        annotation_position="top",
    )
    
    # Corner boundaries
    fig.add_vrect(
        x0=turn1.start_dist,
        x1=turn1.end_dist,
        fillcolor="gray",
        opacity=0.1,
        line_width=0,
        annotation_text=turn1.name,
        annotation_position="top left",
    )
    
    # Apex line
    fig.add_vline(
        x=turn1.apex_dist,
        line_dash="dot",
        line_color="orange",
        opacity=0.5,
        annotation_text="Apex",
        annotation_position="bottom",
    )
    
    fig.update_layout(
        title=f"Throttle Acceptance Visualization - {turn1.name} (Best Lap)",
        xaxis_title="Distance (m)",
        yaxis_title="Lateral G / Throttle (scaled)",
        width=1000,
        height=500,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    
    show_fig(fig)
else:
    print(f"Could not compute throttle acceptance for {turn1.name}")

In [18]:
# Summary statistics table
def compute_summary_stats(stats_df):
    """Compute summary statistics for each segment across all laps."""
    summary = []

    # Braking segments
    for seg_name in stats_df[stats_df["segment_type"] == "braking"]["segment_name"].unique():
        seg_data = stats_df[
            (stats_df["segment_name"] == seg_name) & stats_df["braking_point"].notna()
        ]
        if len(seg_data) > 0:
            summary.append(
                {
                    "Segment": seg_name,
                    "Type": "Braking",
                    "Metric": "Braking Point (m)",
                    "Mean": seg_data["braking_point"].mean(),
                    "Std": seg_data["braking_point"].std(),
                    "Min": seg_data["braking_point"].min(),
                    "Max": seg_data["braking_point"].max(),
                    "Range": seg_data["braking_point"].max() - seg_data["braking_point"].min(),
                    "N": len(seg_data),
                }
            )

    # Corner segments
    for seg_name in stats_df[stats_df["segment_type"] == "corner"]["segment_name"].unique():
        seg_data = stats_df[(stats_df["segment_name"] == seg_name) & stats_df["min_speed"].notna()]
        if len(seg_data) > 0:
            summary.append(
                {
                    "Segment": seg_name,
                    "Type": "Corner",
                    "Metric": "Min Speed (km/h)",
                    "Mean": seg_data["min_speed"].mean(),
                    "Std": seg_data["min_speed"].std(),
                    "Min": seg_data["min_speed"].min(),
                    "Max": seg_data["min_speed"].max(),
                    "Range": seg_data["min_speed"].max() - seg_data["min_speed"].min(),
                    "N": len(seg_data),
                }
            )

    # Acceleration segments
    for seg_name in stats_df[stats_df["segment_type"] == "acceleration"]["segment_name"].unique():
        seg_data = stats_df[
            (stats_df["segment_name"] == seg_name) & stats_df["throttle_point"].notna()
        ]
        if len(seg_data) > 0:
            summary.append(
                {
                    "Segment": seg_name,
                    "Type": "Acceleration",
                    "Metric": "Throttle Point (m)",
                    "Mean": seg_data["throttle_point"].mean(),
                    "Std": seg_data["throttle_point"].std(),
                    "Min": seg_data["throttle_point"].min(),
                    "Max": seg_data["throttle_point"].max(),
                    "Range": seg_data["throttle_point"].max() - seg_data["throttle_point"].min(),
                    "N": len(seg_data),
                }
            )

    return pd.DataFrame(summary)


summary_df = compute_summary_stats(stats_df)
summary_df.style.format(
    {"Mean": "{:.1f}", "Std": "{:.1f}", "Min": "{:.1f}", "Max": "{:.1f}", "Range": "{:.1f}"}
)

,Segment,Type,Metric,Mean,Std,Min,Max,Range,N
0,Turn 1 Braking,Braking,Braking Point (m),576.2,5.4,572.1,586.5,14.4,13
1,Turn 2 Braking,Braking,Braking Point (m),1214.7,5.0,1211.0,1224.5,13.5,13
2,Turn 4 Braking,Braking,Braking Point (m),1903.7,3.7,1901.1,1913.3,12.2,13
3,Turn 5 Braking,Braking,Braking Point (m),2686.3,3.0,2684.0,2693.8,9.8,13
4,Turn 6 Braking,Braking,Braking Point (m),2686.3,3.0,2684.0,2693.8,9.8,13
5,Turn 1,Corner,Min Speed (km/h),64.3,2.7,58.3,68.4,10.1,13
6,Turn 2,Corner,Min Speed (km/h),129.0,4.6,115.8,133.6,17.8,13
7,Turn 3,Corner,Min Speed (km/h),137.0,3.3,131.1,144.2,13.1,13
8,Turn 4,Corner,Min Speed (km/h),87.3,2.9,82.2,92.0,9.8,13
9,Turn 5,Corner,Min Speed (km/h),53.7,2.3,50.4,58.8,8.4,13
